# 18 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~3 min](/lite/notebooks/index.html?path=18-thermal-plume-hdg.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/18-thermal-plume-hdg.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** to
switch story / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — the Beast breathes fire
:class: storytelling

*Press a hot iron to the floor of the Beast's lair and the warm air will not sit still:
it tears loose in a **rising plume**, whips from side to side, and stirs the whole channel.
Today you light that fire — and capture its restless dance with the most elegant tools
in the kit.*
:::

A hot patch on the floor of a tall closed channel drives a **buoyant thermal plume**. Above
a moderate temperature difference the plume stops rising straight: it **meanders and puffs**
— a genuinely **time-dependent** flow. We solve the **Boussinesq** equations with two of
NGSolve's most powerful flow tools:

* an **H(div)-conforming HDG** velocity that is **exactly divergence-free**, and
* a scalar **HDG** temperature,

advanced by an **IMEX** scheme that keeps every implicit operator *constant* — so we
factorise **once** with the CI-safe `sparsecholesky` and reuse it every step.

In non-dimensional form (velocity scaled by the thermal diffusion speed, $T\in[0,1]$):
$$ \tfrac{1}{Pr}\bigl(\partial_t\mathbf u + (\mathbf u\!\cdot\!\nabla)\mathbf u\bigr)
   = \Delta\mathbf u - \nabla p + Ra\,T\,\mathbf e_y,\quad \nabla\!\cdot\!\mathbf u=0,\qquad
   \partial_t T + \mathbf u\!\cdot\!\nabla T = \Delta T . $$

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from ngsolve import *
from netgen.geom2d import SplineGeometry
import numpy as np
from ngsolve.webgui import Draw

Ra, Pr = 1e6, 0.71                                   # Rayleigh & Prandtl numbers
order, maxh, dt = 2, 0.04, 8e-6
W, Hh = 1.0, 2.0                                     # a tall, closed channel

# bottom split into three named pieces so the centre strip can be the hot patch
geo = SplineGeometry()
pts = [geo.AppendPoint(*q) for q in [(0, 0), (0.4, 0), (0.6, 0), (W, 0), (W, Hh), (0, Hh)]]
for a, b, bc in [(0, 1, "bot"), (1, 2, "hot"), (2, 3, "bot"),
                 (3, 4, "wall"), (4, 5, "top"), (5, 0, "wall")]:
    geo.Append(["line", pts[a], pts[b]], bc=bc)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
print(f"channel mesh: {mesh.ne} elements   (hot patch on the floor centre)")

n = specialcf.normal(2)
h = specialcf.mesh_size
def tang(w): return w - (w * n) * n
dS = dx(element_boundary=True)
alpha = 4
walls = "hot|bot|top|wall"                           # no-slip everywhere

## 1. The velocity space — H(div)-conforming HDG

The velocity lives in **`HDiv`** (continuous normal component) paired with a **pressure**
in `L2` one order lower, which makes the discrete velocity **pointwise divergence-free**.
H(div) carries no tangential continuity, so a **facet unknown** (`TangentialFacetFESpace`)
ties the tangential velocity together **weakly**, HDG-style. No-slip walls fix both.

In [ ]:
V    = HDiv(mesh, order=order, dirichlet=walls, dgjumps=True)            # normal-continuous
Vhat = TangentialFacetFESpace(mesh, order=order, dirichlet=walls)        # tangential trace
Q    = L2(mesh, order=order - 1)                                         # pressure
X = V * Vhat * Q
(u, uhat, p), (v, vhat, q) = X.TnT()
gfu = GridFunction(X); velocity = gfu.components[0]
print(f"velocity system: {X.ndof} dofs  (exactly divergence-free)")

## 2. The implicit Stokes operator — assembled once

IMEX treats the *stiff linear* parts **implicitly** and the *cheap nonlinear* parts
(convection, buoyancy) **explicitly**. The implicit momentum part is a **generalised Stokes**
operator — mass $\tfrac{1}{Pr\,\Delta t}$ + viscous HDG + incompressibility — which never
changes, so we factorise it **once**. A tiny $-\varepsilon\,pq$ regularises the zero
pressure block so the symmetric-indefinite system goes through `sparsecholesky` (unit 7).

In [ ]:
eps = 1e-9
a = BilinearForm(X, symmetric=True)
a += 1 / (Pr * dt) * InnerProduct(u, v) * dx
a += InnerProduct(Grad(u), Grad(v)) * dx
a += (-InnerProduct(Grad(u) * n, tang(v - vhat)) - InnerProduct(Grad(v) * n, tang(u - uhat))
      + alpha * order * order / h * InnerProduct(tang(u - uhat), tang(v - vhat))) * dS
a += (-div(u) * q - div(v) * p - eps * p * q) * dx
a.Assemble()
ainv = a.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")            # factor ONCE, reuse
mass_u = BilinearForm(1 / (Pr * dt) * InnerProduct(u, v) * dx, symmetric=True).Assemble()

## 3. The temperature — a scalar HDG

Temperature lives in **`L2`** with a **facet** unknown for the HDG diffusion (also constant
in time, also factored once). The **hot patch** is $T=1$, the **ceiling** $T=0$ (Dirichlet
on the facet trace); side walls and the rest of the floor are **insulated** (natural
Neumann). One `BoundaryCF` sets both Dirichlet values at once — *two* separate `Set` calls
would cancel, since each zeroes the whole vector first.

In [ ]:
Wt   = L2(mesh, order=order)
What = FacetFESpace(mesh, order=order, dirichlet="hot|top")
Y_ = Wt * What
(T, That), (s, shat) = Y_.TnT()
gfT = GridFunction(Y_); temp = gfT.components[0]

aT = BilinearForm(Y_, symmetric=True)
aT += 1 / dt * T * s * dx + Grad(T) * Grad(s) * dx
aT += (-Grad(T) * n * (s - shat) - Grad(s) * n * (T - That)
       + alpha * order * order / h * (T - That) * (s - shat)) * dS
aT.Assemble()
aTinv = aT.mat.Inverse(Y_.FreeDofs(), inverse="sparsecholesky")
mass_T = BilinearForm(1 / dt * T * s * dx, symmetric=True).Assemble()
gfT.components[1].Set(mesh.BoundaryCF({"hot": 1, "top": 0}),
                     definedon=mesh.Boundaries("hot|top"))

## 4. The explicit pieces — convection & buoyancy

Convection is **explicit**: upwind-DG for the temperature transport $\mathbf u\!\cdot\!\nabla T$,
and the velocity self-advection $(\mathbf u\!\cdot\!\nabla)\mathbf u$ (the element-local form,
as in NGSolve's Navier–Stokes demo). Buoyancy $Ra\,T\,\mathbf e_y$ enters the momentum
right-hand side. Because only the thin plume moves fast (the bulk of the channel is nearly
still), the explicit-convection CFL limit on `dt` stays mild despite the high Rayleigh number.

In [ ]:
convT = BilinearForm(Y_, nonassemble=True)                              # temperature transport
uTn = velocity * n
convT += -T * (velocity * Grad(s)) * dx
convT += uTn * IfPos(uTn, T, T.Other()) * (s - s.Other()) * dx(skeleton=True)

convU = BilinearForm(X, nonassemble=True)                              # velocity self-advection
convU += 1 / Pr * InnerProduct(Grad(u) * u, v) * dx

buoyancy = LinearForm(Ra * temp * v[1] * dx)                           # Ra·T·e_y on the rhs
resT = gfT.vec.CreateVector(); resU = gfu.vec.CreateVector()

## 5. Step in time — and watch the plume whip

One IMEX step: advance the temperature (explicit upwind transport on the right, implicit
diffusion via `aTinv`, Dirichlet kept by a residual update), then the velocity (explicit
self-advection + the buoyancy `Bmat·T` from the fresh temperature, implicit Stokes via
`ainv`). We snapshot the temperature for an animation and probe the velocity above the
patch — once the plume starts meandering, that probe **oscillates**.

In [ ]:
tend = 0.04
nsteps = int(tend / dt + 0.5)
gx, gy = np.linspace(0, W, 70), np.linspace(0, Hh, 140)                 # animation sample grid
mips = [mesh(xx, yy) for yy in gy for xx in gx]                         # mapped points, found ONCE
probe = mesh(0.5, 1.2)
frames, vx_t, vy_t, ts = [], [], [], []
with TaskManager():
    for step in range(1, nsteps + 1):
        convT.Apply(gfT.vec, resT)                                     # explicit transport
        gfT.vec.data += aTinv * ((mass_T.mat * gfT.vec - resT).Evaluate() - aT.mat * gfT.vec)
        convU.Apply(gfu.vec, resU)                                     # explicit self-advection
        buoyancy.Assemble()                                            # explicit buoyancy Ra·T·e_y
        gfu.vec.data = ainv * (mass_u.mat * gfu.vec + buoyancy.vec - resU)  # implicit Stokes
        if step % 160 == 0:
            frames.append(np.array([temp(mp) for mp in mips]).reshape(len(gy), len(gx)))
            vv = velocity(probe); ts.append(step * dt); vx_t.append(vv[0]); vy_t.append(vv[1])
print(f"done: {nsteps} steps,  ‖div u‖ = {sqrt(Integrate(div(velocity)**2, mesh)):.1e}  "
      f"(exactly div-free),  collected {len(frames)} frames")

## 6. The result — a meandering, puffing plume

The plume rises straight from the hot patch, mushrooms at the top, then loses its symmetry
and **whips from side to side** — an unsteady, self-sustained dance. (Press play.)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(2.6, 5.0))                              # gx, gy from the time loop
def draw(i):
    ax.clear()
    ax.contourf(gx, gy, frames[i], levels=np.linspace(0, 1, 21), cmap="hot")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"t = {ts[i]:.3f}")
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=150)
plt.close(fig)
HTML(anim.to_jshtml())

The vertical velocity probed just above the patch keeps **oscillating** — the signature of
the unsteady plume (a steady plume would flatline):

In [ ]:
plt.figure(figsize=(8, 2.6))
plt.plot(ts, vy_t, label="$v_y$"); plt.plot(ts, vx_t, label="$v_x$")
plt.axhline(0, ls="--", c="gray", lw=0.8); plt.legend()
plt.xlabel("time"); plt.ylabel("velocity at (0.5, 1.2)")
plt.title("the plume keeps meandering (unsteady)"); plt.grid(alpha=0.3); plt.tight_layout()

In [ ]:
Draw(temp, mesh, "temperature", min=0, max=1, autoscale=False)

In [ ]:
Draw(velocity, mesh, "velocity", vectors={"grid_size": 30})

:::{dropdown} 📚 Further reading
:class: further-reading

- **H(div)-conforming HDG for incompressible flow** — exactly divergence-free, pressure-robust
  (Lehrenfeld & Schöberl). The ngs24 CFD tutorials this notebook is built on:
  [normal-continuous HDG](https://docu.ngsolve.org/ngs24/CFD/stokes_hdg.html),
  [scalar HDG tricks](https://docu.ngsolve.org/ngs24/CFD/hdg_tricks.html),
  [H(div)-HDG tricks](https://docu.ngsolve.org/ngs24/CFD/hdivhdg_tricks.html) and
  [instationary Navier–Stokes (HDG)](https://docu.ngsolve.org/ngs24/CFD/navierstokes_hdg.html).
- **Incompressible Navier–Stokes in NGSolve** — i-tutorial
  [3.2](https://docu.ngsolve.org/latest/i-tutorials/unit-3.2-navierstokes/navierstokes.html),
  whose explicit-convection IMEX splitting we follow here.
- **Buoyant convection benchmarks** — the steady [de Vahl Davis](https://doi.org/10.1002/fld.1650030305)
  cavity and Rayleigh–Bénard convection.
:::

:::{dropdown} 🧠 Quiz — why factor only once?
:class: quiz
Because **IMEX** puts everything that changes from step to step — the nonlinear convection,
the buoyancy load $Ra\,T\,\mathbf e_y$ — on the **right-hand side**, while the implicit
operators (generalised Stokes and temperature diffusion) depend only on `dt`, the mesh and
the parameters, all **fixed**. A constant matrix means **one** `sparsecholesky` factorisation,
reused as a cheap back-substitution every step — fast *and* CI-safe.
:::

**That is the toolkit:** an exactly divergence-free H(div)-HDG velocity, a scalar HDG
temperature, coupled through buoyancy and marched with a factor-once IMEX scheme — enough to
make a thermal plume dance. The expedition continues. ☕

In [ ]:
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _nb, _title = "19-outlook-unfitted", "19 · Outlook — unfitted FEM with ngsxfem 🫧"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next:** [" + _title + "](" + _u + ")"))